In [6]:
"""
most elementary version of Recurrent Neural Network
"""

import random


data: list[list[float]] = [
    [0.0, 0.0, 0.0, 0.0],  # cheap things keep being cheap
    [1.0, 1.0, 1.0, 1.0],  # expensise things keep being expensive
    [0.0, 0.5, 0.6, 1.0],  # growth trend
    [1.0, 0.5, 0.3, 0.0],  # decline trend
]

X = [row[:2] for row in data]
Y = [row[-1] for row in data]
seq_len = len(X[0])
batch_len = len(data)

rfl = random.random
params = rfl(), rfl(), rfl(), rfl(), rfl()
w1, b1, w2, w3, b2 = params


def relu(x): return max(0, x)


def loss_fn(inputs, y):
    xs = {}
    inp_layers = {}
    relu_inps = {}
    relu_outs = {}
    recurr_layers = {}

    dw1, db1, dw2, dw3, db2 = 0.0, 0.0, 0.0, 0.0, 0.0

    prev_h = 0.0
    res = 0.0
    loss = 0.0
    recurr_layers[-1] = 0.0

    # forward pass
    last_ts = seq_len - 1
    for i in range(seq_len):
        xs[i] = inputs[i]

        inp_layers[i] = xs[i] * w1
        relu_inps[i] = inp_layers[i] + recurr_layers[i - 1] + b1
        relu_outs[i] = relu(relu_inps[i])
        recurr_layers[i] = relu_outs[i] * w2

        if i < last_ts:
            continue

        res = relu_outs[i] * w3 + b2
        # decomposition of the loss function, just to simplify backprop for me
        u = res - y
        loss = u ** 2

    # backward pass
    du, dres, dy = None, None, None
    drelu_out_j = 0.0
    for j in reversed(range(seq_len)):
        dloss = 1
        drelu_local_j = 1.0 if relu_inps[j] > 0 else 0.0
        # incoming grad to relu_outs[j] from future timestep(s)

        if j == last_ts:
            du = dloss * 2 * u
            dres = du
            dy = -du
            db2 += dres * 1
            dw3 += dres * relu_outs[j]
            drelu_out_j += dres * w3
        drelu_inp_j = drelu_out_j * drelu_local_j

        dinp_layer_j = drelu_inp_j * 1
        db1 += drelu_inp_j * 1
        dw1 += dinp_layer_j * xs[j]

        dw2 += drelu_inp_j * (relu_outs[j-1] if j > 0 else 0.0)

        drelu_out_j = drelu_inp_j * w2

    return loss, res, [dw1, dw2, dw3, db1, db2]


N_EPOCH = 100
lr = 1e-1
for i in range(N_EPOCH):
    loss = 0.0
    for b in range(batch_len):
        x_batch = X[b]
        y_batch = Y[b]
        loss_, res, grads = loss_fn(x_batch, y_batch)
        loss += loss_
        [dw1, dw2, dw3, db1, db2] = grads
        w1 += -lr * dw1
        w2 += -lr * dw2
        w3 += -lr * dw3
        b1 += -lr * db1
        b2 += -lr * db2
    
    loss /= batch_len
    print(i, loss)

0 0.5006974683927236
1 0.358681020771343
2 0.34656924208224654
3 0.33810701342268745
4 0.3318152182277951
5 0.32688804256526677
6 0.322875095633218
7 0.3195118325038573
8 0.31663503874270926
9 0.3141394299684477
10 0.3119540595359174
11 0.3100289472432387
12 0.3083272750805655
13 0.30682071999084093
14 0.30539686461983534
15 0.3033716575453716
16 0.3027042977174688
17 0.3020831888308803
18 0.3014956995416292
19 0.2997087358786287
20 0.30048323110418984
21 0.29999084553258765
22 0.29949001476225434
23 0.2963824458367741
24 0.2989002936506383
25 0.2983326379583815
26 0.29778982908792523
27 0.29727295500943274
28 0.2936486651404058
29 0.29683908775289075
30 0.29619390875242385
31 0.2956321645878383
32 0.29512800134660816
33 0.2902145488943783
34 0.29462757241520504
35 0.29394482273903316
36 0.2933643301252199
37 0.2928342696831692
38 0.2857588860586834
39 0.29207212659528786
40 0.29132488306442683
41 0.2906463642001172
42 0.2899837218425798
43 0.28932326796573504
44 0.2816861190338381
45 